# P3-06 · Train D-FINE (single class `tb_lesion`) on a T4, with auto-resume

Runs **one** config per session: `dfine_512.yaml` or `dfine_640.yaml` (identical except input size).
Train split only for training, development only for evaluation; nothing else is loaded.

**Before the first run** follow `docs/p3_kaggle_runbook.md` sections 0–4 (phone verification, rclone secret
`RCLONE_CONF`, optional `GITHUB_TOKEN`). Session options: **GPU T4**, **Internet On**.
Data: add P1's processed data (C01 split CSV, C04 PNGs + manifest, C05 parquet, C06 COCO) as a **private**
Kaggle dataset via *+ Add Input* and set `DATA_ROOT` below.

For the full run use **Save Version → Save & Run All**. If the session dies, run this notebook again
unchanged: it resumes from the last checkpoint on Drive (at most one epoch lost).

Preprocessing is **PROVISIONAL** (margin 0.10, CLAHE off) until P2 locks D08/D09. Re-run both sizes after
that: the run folder name changes automatically (`..._PROVISIONAL` vs locked), so the runs never mix.

In [ ]:
CONFIG = "dfine_512.yaml"                      # or "dfine_640.yaml"
REPO = "github.com/<org>/cv_proj.git"            # TODO: team repo
BRANCH = "main"
DATA_ROOT = "/kaggle/input/<tb-processed-v1>"    # TODO: folder that contains data/splits, data/processed, data/boxes
WORK = "/kaggle/working/cv_proj"

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
try:
    url = f"https://x-access-token:{UserSecretsClient().get_secret('GITHUB_TOKEN')}@{REPO}"
except Exception:
    url = f"https://{REPO}"
if not os.path.isdir(WORK):
    subprocess.run(["git", "clone", "-q", "--recurse-submodules", "-b", BRANCH, url, WORK], check=True)
os.chdir(WORK)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
print("D-FINE:", subprocess.run(["git", "-C", "third_party/D-FINE", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements/p3.txt
import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
# Drive via rclone (Kaggle). On Colab, CheckpointStore mounts Drive itself; skip this cell.
!test -x /usr/local/bin/rclone || (wget -q https://downloads.rclone.org/rclone-current-linux-amd64.zip -O /tmp/rclone.zip   && unzip -q -o /tmp/rclone.zip -d /tmp && cp /tmp/rclone-*-linux-amd64/rclone /usr/local/bin/)
import sys; sys.path.insert(0, "src")
from p3_ops.cloud import setup_rclone_from_kaggle_secret
print("rclone configured:", setup_rclone_from_kaggle_secret())
!rclone lsd gdrive:tb_checkpoints

In [ ]:
os.environ["TB_DATA_ROOT"] = DATA_ROOT
for f in ("data/splits/tbx11k_splits_v1.csv", "data/processed/v1/manifest.parquet",
          "data/processed/v1/lung_roi.parquet", "data/boxes/v1"):
    print(f, os.path.exists(os.path.join(DATA_ROOT, f)))
for d in ("decisions/p2_clahe.json", "decisions/p2_crop_margin.json"):
    print(d, "exists" if os.path.exists(d) else "missing -> PROVISIONAL preprocessing")

In [ ]:
!PYTHONPATH=src python -m p3_det.train --config {CONFIG}

In [ ]:
# Summary + checkpoint hashes (written into the run folder and copied to Drive)
import glob, hashlib, json, pandas as pd
from p3_det.train.config import load_run_config
name = load_run_config(CONFIG).run_name
rd = sorted(glob.glob(f"/kaggle/working/checkpoints/{name}_m*"))[-1]
print(rd)
display(pd.read_csv(f"{rd}/train_log.csv").tail(5))
lines = []
for p in sorted(glob.glob(f"{rd}/ckpt_*.pt")):
    h = hashlib.sha256(open(p, "rb").read()).hexdigest()
    lines.append(f"{h}  {os.path.basename(p)}")
open(f"{rd}/checkpoints_sha256.txt", "w").write("
".join(lines) + "
")
print("
".join(lines))
!rclone copy {rd}/checkpoints_sha256.txt gdrive:tb_checkpoints/$(basename {rd})

## Bring back (after BOTH sizes finished)

From Drive `tb_checkpoints/<run folder>/` for **each** run folder (e.g. `dfine_s_512_m0.10_clahe0_PROVISIONAL`):

| File | Into the repo at |
|---|---|
| `run_meta.json`, `train_log.csv`, `dev_ground_truth.json`, `dev_detections_last.json`, `dev_detections_best.json`, `best.json`, `checkpoints_sha256.txt` | `artifacts/p3/det/runs/<run folder>/` |
| `ckpt_epochNNNN.pt`, `ckpt_best.pt` | **stay on Drive** (only their SHA-256 comes back) |

Then on the laptop: `python -m p3_det.select_size compare --run512 artifacts/p3/det/runs/<512 folder> --run640 artifacts/p3/det/runs/<640 folder>`